## **Etape 2 - Meteo, scoring et carte Top 5**

Objectifs:
1. Charger les villes geocodees depuis `outputs/data/cities_geocoded.csv`.
2. Recuperer la meteo 7 jours via OpenWeatherMap One Call.
3. Construire un dataset meteo propre et un score de classement.
4. Sauvegarder les resultats en CSV.
5. Visualiser les 5 meilleures destinations sur une carte Plotly.

### **Import librairies necessaire**

In [10]:
from pathlib import Path
import os
import time
import requests
import pandas as pd
import numpy as np
import plotly.express as px
from dotenv import load_dotenv

### **Chargement des variables env et cle API**

In [11]:
load_dotenv(override=True)

TARGET_TEMP_C = 22
API_KEY = os.getenv("OPEN_WEATHER_MAP_API_KEY")
if not API_KEY:
    raise ValueError(
        "Cle API manquante: ajoute OPEN_WEATHER_MAP_API_KEY dans ton fichier .env"
    )

print("Cle API chargee: OK")

Cle API chargee: OK


### **Chargement des villes geocodees**

In [12]:
cities_path = Path("../outputs/data/cities_geocoded.csv")
if not cities_path.exists():
    raise FileNotFoundError(
        "Fichier introuvable: ../outputs/data/cities_geocoded.csv. Execute d'abord le notebook 01."
    )

cities_df = pd.read_csv(cities_path)

if "found" in cities_df.columns:
    cities_df = cities_df[cities_df["found"] == True].copy()  # noqa: E712

required_cols = ["city_id", "city_name", "latitude", "longitude"]
missing_cols = [c for c in required_cols if c not in cities_df.columns]
if missing_cols:
    raise ValueError(f"Colonnes manquantes dans cities_geocoded.csv: {missing_cols}")

cities_df = cities_df[required_cols].dropna().reset_index(drop=True)
print(f"Nombre de villes retenues: {len(cities_df)}")
cities_df.head()

Nombre de villes retenues: 35


,city_id,city_name,latitude,longitude
0,1,Mont Saint Michel,48.635954,-1.511460
1,2,St Malo,48.649518,-2.026041
2,3,Bayeux,49.276462,-0.702474
3,4,Le Havre,49.493898,0.107973
4,5,Rouen,49.440459,1.093966


### **Definition de la fonction d'appel OpenWeatherMap (7 jours)**

In [13]:
ONE_CALL_URL = "https://api.openweathermap.org/data/3.0/onecall"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (contact: raphael@nexthope.net)"
}

def get_daily_forecast(lat: float, lon: float, api_key: str, n_days: int = 7):
    params = {
        "lat": lat,
        "lon": lon,
        "exclude": "current,minutely,hourly,alerts",
        "appid": api_key,
        "units": "metric",
        "lang": "fr",
    }

    response = requests.get(ONE_CALL_URL, params=params, headers=HEADERS, timeout=30)
    response.raise_for_status()

    payload = response.json()
    daily = payload.get("daily", [])
    return daily[:n_days]

In [14]:
get_daily_forecast(48.8566, 2.3522, API_KEY)

[{'dt': 1776942000,
  'sunrise': 1776919475,
  'sunset': 1776970377,
  'moonrise': 1776936300,
  'moonset': 1776907560,
  'moon_phase': 0.22,
  'summary': 'There will be clear sky today',
  'temp': {'day': 19.92,
   'min': 7.89,
   'max': 21.55,
   'night': 15.22,
   'eve': 19.98,
   'morn': 7.89},
  'feels_like': {'day': 18.7, 'night': 14.21, 'eve': 18.87, 'morn': 5.96},
  'pressure': 1026,
  'humidity': 28,
  'dew_point': 0.12,
  'wind_speed': 5.58,
  'wind_deg': 78,
  'wind_gust': 13.33,
  'weather': [{'id': 800,
    'main': 'Clear',
    'description': 'ciel dégagé',
    'icon': '01d'}],
  'clouds': 0,
  'pop': 0,
  'uvi': 4.84},
 {'dt': 1777028400,
  'sunrise': 1777005764,
  'sunset': 1777056866,
  'moonrise': 1777027620,
  'moonset': 1776995940,
  'moon_phase': 0.25,
  'summary': 'Expect a day of partly cloudy with clear spells',
  'temp': {'day': 19.32,
   'min': 9.85,
   'max': 21.38,
   'night': 14.89,
   'eve': 19.89,
   'morn': 9.85},
  'feels_like': {'day': 17.99, 'night': 1

### **Recuperation & Aggregation meteo par ville**

In [15]:
weather_rows = []

for idx, row in cities_df.iterrows():
    city_id = row["city_id"]
    city_name = row["city_name"]
    lat = float(row["latitude"])
    lon = float(row["longitude"])

    try:
        daily_data = get_daily_forecast(lat, lon, API_KEY, n_days=7)

        if len(daily_data) == 0:
            weather_rows.append(
                {
                    "city_id": city_id,
                    "city_name": city_name,
                    "latitude": lat,
                    "longitude": lon,
                    "days_count": 0,
                    "avg_temp_7d": np.nan,
                    "avg_humidity_7d": np.nan,
                    "total_rain_mm_7d": np.nan,
                    "avg_pop_7d": np.nan,
                    "status": "NO_DAILY_DATA",
                }
            )
            print(f"[{idx+1:02d}/{len(cities_df)}] {city_name}: NO_DAILY_DATA")
            time.sleep(0.2)
            continue

        temps = [d.get("temp", {}).get("day", np.nan) for d in daily_data]
        humidities = [d.get("humidity", np.nan) for d in daily_data]
        rains = [d.get("rain", 0.0) for d in daily_data]
        pops = [d.get("pop", np.nan) for d in daily_data]

        weather_rows.append(
            {
                "city_id": city_id,
                "city_name": city_name,
                "latitude": lat,
                "longitude": lon,
                "days_count": len(daily_data),
                "avg_temp_7d": float(np.nanmean(temps)),
                "avg_humidity_7d": float(np.nanmean(humidities)),
                "total_rain_mm_7d": float(np.nansum(rains)),
                "avg_pop_7d": float(np.nanmean(pops)),
                "status": "OK",
            }
        )
        print(f"[{idx+1:02d}/{len(cities_df)}] {city_name}: OK")

    except Exception as exc:
        weather_rows.append(
            {
                "city_id": city_id,
                "city_name": city_name,
                "latitude": lat,
                "longitude": lon,
                "days_count": 0,
                "avg_temp_7d": np.nan,
                "avg_humidity_7d": np.nan,
                "total_rain_mm_7d": np.nan,
                "avg_pop_7d": np.nan,
                "status": f"ERROR: {exc}",
            }
        )
        print(f"[{idx+1:02d}/{len(cities_df)}] {city_name}: ERROR")

    # petite pause pour eviter trop de requetes en meme temps
    time.sleep(0.2)

weather_df = pd.DataFrame(weather_rows)
weather_df.head()

[01/35] Mont Saint Michel: OK
[02/35] St Malo: OK
[03/35] Bayeux: OK
[04/35] Le Havre: OK
[05/35] Rouen: OK
[06/35] Paris: OK
[07/35] Amiens: OK
[08/35] Lille: OK
[09/35] Strasbourg: OK
[10/35] Chateau du Haut Koenigsbourg: OK
[11/35] Colmar: OK
[12/35] Eguisheim: OK
[13/35] Besancon: OK
[14/35] Dijon: OK
[15/35] Annecy: OK
[16/35] Grenoble: OK
[17/35] Lyon: OK
[18/35] Gorges du Verdon: OK
[19/35] Bormes les Mimosas: OK
[20/35] Cassis: OK
[21/35] Marseille: OK
[22/35] Aix en Provence: OK
[23/35] Avignon: OK
[24/35] Uzes: OK
[25/35] Nimes: OK
[26/35] Aigues Mortes: OK
[27/35] Saintes Maries de la mer: OK
[28/35] Collioure: OK
[29/35] Carcassonne: OK
[30/35] Ariege: OK
[31/35] Toulouse: OK
[32/35] Montauban: OK
[33/35] Biarritz: OK
[34/35] Bayonne: OK
[35/35] La Rochelle: OK


,city_id,city_name,latitude,longitude,days_count,avg_temp_7d,avg_humidity_7d,total_rain_mm_7d,avg_pop_7d,status
0,1,Mont Saint Michel,48.635954,-1.511460,7,17.590000,52.857143,0.00,0.000000,OK
1,2,St Malo,48.649518,-2.026041,7,14.870000,65.428571,0.00,0.000000,OK
2,3,Bayeux,49.276462,-0.702474,7,15.278571,61.285714,0.57,0.028571,OK
3,4,Le Havre,49.493898,0.107973,7,14.660000,60.285714,0.30,0.044286,OK
4,5,Rouen,49.440459,1.093966,7,16.645714,52.714286,0.66,0.115714,OK


### **Logique de scoring meteo**

In [16]:
scored_df = weather_df[weather_df["status"] == "OK"].copy()

# Distance a la temperature cible (TARGET_TEMP_C degres): plus c'est faible, mieux c'est
scored_df["temp_distance_to_22"] = (scored_df["avg_temp_7d"] - TARGET_TEMP_C).abs()


def min_max(series: pd.Series):
    s_min = series.min()
    s_max = series.max()
    if pd.isna(s_min) or pd.isna(s_max) or s_min == s_max:
        return pd.Series([1.0] * len(series), index=series.index)
    return (series - s_min) / (s_max - s_min)


# Tous ces indicateurs doivent etre minimises, donc score inverse (1 - normalise)
scored_df["s_rain"] = 1 - min_max(scored_df["total_rain_mm_7d"])
scored_df["s_pop"] = 1 - min_max(scored_df["avg_pop_7d"])
scored_df["s_humidity"] = 1 - min_max(scored_df["avg_humidity_7d"])
scored_df["s_temp"] = 1 - min_max(scored_df["temp_distance_to_22"])

# Ponderation meteo (ajustable)
weights = {
    "s_rain": 0.35,
    "s_pop": 0.30,
    "s_temp": 0.25,
    "s_humidity": 0.10,
}

scored_df["weather_score"] = (
    scored_df["s_rain"] * weights["s_rain"]
    + scored_df["s_pop"] * weights["s_pop"]
    + scored_df["s_temp"] * weights["s_temp"]
    + scored_df["s_humidity"] * weights["s_humidity"]
) * 100

scored_df["weather_rank"] = scored_df["weather_score"].rank(ascending=False, method="dense").astype(int)
scored_df = scored_df.sort_values("weather_score", ascending=False).reset_index(drop=True)

scored_df[["city_id", "city_name", "weather_score", "weather_rank"]].head(10)

,city_id,city_name,weather_score,weather_rank
0,16,Grenoble,97.621986,1
1,29,Carcassonne,90.066994,2
2,17,Lyon,88.767669,3
3,22,Aix en Provence,88.377337,4
4,6,Paris,85.875572,5
5,15,Annecy,85.703680,6
6,31,Toulouse,81.992739,7
7,32,Montauban,80.741092,8
8,19,Bormes les Mimosas,80.305550,9
9,1,Mont Saint Michel,79.545235,10


### **Export datasets meteo en CSV**

In [17]:
output_data_dir = Path("../outputs/data")
output_data_dir.mkdir(parents=True, exist_ok=True)

raw_output_file = output_data_dir / "cities_weather_7d_raw.csv"
scored_output_file = output_data_dir / "cities_weather_scored.csv"

weather_df.to_csv(raw_output_file, index=False)
scored_df.to_csv(scored_output_file, index=False)

print(f"Export brut: {raw_output_file.resolve()}")
print(f"Export score: {scored_output_file.resolve()}")

Export brut: /home/raphael/DataProjects/jedha-certif/projet-kayak-bloc1/outputs/data/cities_weather_7d_raw.csv
Export score: /home/raphael/DataProjects/jedha-certif/projet-kayak-bloc1/outputs/data/cities_weather_scored.csv


### **Visualiser les 5 meilleures destinations**

In [18]:
import os

top5_df = scored_df.nsmallest(5, "weather_rank").copy()

fig = px.scatter_map(
    top5_df,
    lat="latitude",
    lon="longitude",
    hover_name="city_name",
    hover_data={
        "weather_score": ":.2f",
        "avg_temp_7d": ":.1f",
        "total_rain_mm_7d": ":.1f",
        "avg_pop_7d": ":.2f",
        "avg_humidity_7d": ":.1f",
        "weather_rank": True,
    },
    color="weather_score",
    size="weather_score",
    zoom=4.7,
    height=600,
    title="Top 5 destinations selon le score meteo (7 jours)",
    color_continuous_scale="Viridis",
)

fig.update_layout(margin={"r": 0, "t": 60, "l": 0, "b": 0})
fig.show()

output_map_dir = Path("../outputs/maps")
output_map_dir.mkdir(parents=True, exist_ok=True)
map_output_png = output_map_dir / "top5_destinations_weather_map.png"

# Save as PNG (static image)
# Note: Requires kaleido package (`pip install -U kaleido`)
try:
    fig.write_image(map_output_png, scale=2)
    print(f"Image PNG exportee: {map_output_png.resolve()}")
except Exception as e:
    print(f"Export PNG echouée: {e}")


Image PNG exportee: /home/raphael/DataProjects/jedha-certif/projet-kayak-bloc1/outputs/maps/top5_destinations_weather_map.png


### **Prerequis et execution (Pipenv)**

#### *Prerequis*
- Le notebook 01 execute (fichier `outputs/data/cities_geocoded.csv` present)
- La variable `OPEN_WEATHER_MAP_API_KEY` renseignee dans `.env`